# ecoli-0-1-3-7_vs_2-6: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('09_ecoli-0-1-3-7_vs_2-6_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 4


,classifier,method,seed,fold,error
112,svm,adasyn,137,2,Not any neigbours belong to the majority class...
121,knn,adasyn,137,2,Not any neigbours belong to the majority class...
256,rf,adasyn,137,2,RuntimeError: Not any neigbours belong to the ...
265,lr,adasyn,137,2,RuntimeError: Not any neigbours belong to the ...


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,74.142,73.333,80.000,70.0,100.000,85.000,71.810,82.684,73.738
smote,74.051,71.333,76.667,70.0,99.818,84.909,73.505,93.249,72.692
bsmote,74.142,73.333,80.000,70.0,100.000,85.000,73.950,90.985,73.738
ros,72.283,43.000,40.333,70.0,96.364,83.182,51.997,92.136,57.642
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,74.142,73.333,80.000,70.0,100.000,85.000,72.231,91.027,73.738
casb_relaxed,74.142,73.333,80.000,70.0,100.000,85.000,74.123,94.700,73.738
casb_edited,73.777,62.667,63.333,70.0,99.273,84.636,73.123,93.332,68.222
casb_ros,74.142,73.333,80.000,70.0,100.000,85.000,72.251,91.024,73.738


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,73.645,59.333,53.333,70.0,98.902,84.451,70.714,84.722,66.489
smote,72.161,36.778,27.357,70.0,95.993,82.997,49.048,84.266,54.469
bsmote,72.716,46.111,37.857,70.0,97.084,83.542,52.381,84.357,59.414
ros,71.968,36.000,26.667,70.0,95.630,82.815,35.714,83.902,53.984
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,71.493,32.939,24.278,70.0,94.721,82.360,35.714,83.902,52.216
casb_relaxed,72.437,41.222,31.548,70.0,96.539,83.269,55.714,84.448,56.830
casb_edited,71.299,32.000,23.500,70.0,94.357,82.178,49.048,84.266,51.649
casb_ros,71.968,36.000,26.667,70.0,95.630,82.815,35.714,83.902,53.984


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,14.142,13.333,20.000,10.0,100.000,55.000,76.944,97.259,13.738
smote,73.894,67.667,68.333,70.0,99.451,84.726,73.988,90.258,70.780
bsmote,74.011,70.000,70.000,70.0,99.630,84.815,74.385,90.803,72.005
ros,64.011,60.000,60.000,60.0,99.630,79.815,73.036,86.618,62.005
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,74.011,70.000,70.000,70.0,99.630,84.815,73.383,89.345,72.005
casb_relaxed,74.011,70.000,70.000,70.0,99.630,84.815,74.083,93.165,72.005
casb_edited,74.011,70.000,70.000,70.0,99.630,84.815,74.595,94.712,72.005
casb_ros,74.011,70.000,70.000,70.0,99.630,84.815,72.786,85.887,72.005


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,74.011,70.000,70.000,70.0,99.630,84.815,78.667,96.542,72.005
smote,72.716,48.333,42.000,70.0,97.084,83.542,77.000,96.357,60.524
bsmote,73.182,53.333,46.667,70.0,97.993,83.997,77.833,96.449,63.258
ros,71.359,31.463,21.444,70.0,94.350,82.175,76.000,96.172,51.411
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,72.908,44.762,34.667,70.0,97.448,83.724,77.000,96.357,58.835
casb_relaxed,72.716,48.000,41.667,70.0,97.084,83.542,77.000,96.357,60.358
casb_edited,72.716,48.000,41.667,70.0,97.084,83.542,74.361,95.524,60.358
casb_ros,72.908,44.762,34.667,70.0,97.448,83.724,77.000,96.357,58.835


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,58.985,54.000,55.833,55.0,99.633,77.316,74.534,90.302,56.493
smote,73.205,56.028,53.589,70.0,98.087,84.043,68.385,91.032,64.616
bsmote,73.513,60.694,58.631,70.0,98.677,84.338,69.637,90.649,67.104
ros,69.905,42.616,37.111,67.5,96.493,81.997,59.187,89.707,56.260
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,73.139,55.259,52.236,70.0,97.949,83.975,64.582,90.158,64.199
casb_relaxed,73.327,58.139,55.804,70.0,98.313,84.157,70.230,92.168,65.733
casb_edited,72.951,53.167,49.625,70.0,97.586,83.793,67.782,91.958,63.059
casb_ros,73.257,56.024,52.833,70.0,98.177,84.088,64.438,89.293,64.640


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,result
0,gmean,Incomplete coverage
1,f1,Incomplete coverage
2,precision,Incomplete coverage
3,recall,Incomplete coverage


## knn: per-metric family comparison

,metric,result
0,gmean,Incomplete coverage
1,f1,Incomplete coverage
2,precision,Incomplete coverage
3,recall,Incomplete coverage


## rf: per-metric family comparison

,metric,result
0,gmean,Incomplete coverage
1,f1,Incomplete coverage
2,precision,Incomplete coverage
3,recall,Incomplete coverage


... (3 outputs left out)

,metric,result
0,gmean,Incomplete coverage
1,f1,Incomplete coverage
2,precision,Incomplete coverage
3,recall,Incomplete coverage


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,NaN
1,42,1,svm,smote,0.50,0.990867,0.800000,0.666667,1.0,0.981818,0.990909,1.000000,1.000000,0.895434,NaN
2,42,1,svm,bsmote,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,NaN
3,42,1,svm,ros,0.50,0.943880,0.400000,0.250000,1.0,0.890909,0.945455,0.416667,0.963636,0.671940,NaN
4,42,1,svm,adasyn,0.50,0.990867,0.800000,0.666667,1.0,0.981818,0.990909,1.000000,1.000000,0.895434,NaN
5,42,1,svm,casb,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,NaN
6,42,1,svm,casb_relaxed,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,NaN
7,42,1,svm,casb_edited,0.50,0.990867,0.800000,0.666667,1.0,0.981818,0.990909,1.000000,1.000000,0.895434,NaN
8,42,1,svm,casb_ros,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,NaN
9,42,1,knn,original,0.50,0.990867,0.800000,0.666667,1.0,0.981818,0.990909,1.000000,1.000000,0.895434,NaN
